# Millionaire Project - Large Experiments

Use this notebook in Google Colab when your local machine is too slow for repeated seeds, larger backtests, or parameter sweeps.

Run the cells from top to bottom. Start with the small settings first, then scale up.

## 1. Get The Project

This clones the GitHub repository into the Colab runtime. If you opened this notebook directly from GitHub, you still need this step because Colab loads the notebook file, not the whole repo.

In [ ]:
!git clone https://github.com/charlesclothilde-cmd/MillionaireProject.git
%cd MillionaireProject
!pip install -q -r requirements.txt

Optional: mount Google Drive if you want outputs to survive after the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Load The Data And Helpers

In [ ]:
from pathlib import Path
import pandas as pd

from lottery_data import (
    load_draws,
    data_freshness_label,
    run_repeated_backtests,
    summarise_repeated_backtests,
    run_parameter_sweep,
    backtest_model_ablation,
    summarise_backtest,
    summarise_prize_tier_values,
)

df = load_draws('euromillions_full_2004_2026.csv')
print(data_freshness_label(df))
print(f'Draws loaded: {len(df):,}')

output_dir = Path('/content/drive/MyDrive/MillionaireProject_outputs')
output_dir.mkdir(parents=True, exist_ok=True)
print(f'Outputs will be saved to: {output_dir}')

## 3. Smoke Test

Run this first. If this works, the full project is available in Colab.

In [ ]:
smoke = run_repeated_backtests(
    df,
    seeds=list(range(3)),
    training_window=300,
    test_draws=25,
    recent_draw_count=100,
    simulations=50,
    half_life=75,
)

smoke_summary = summarise_repeated_backtests(smoke)
smoke_summary

## 4. Recommended Large Run

This is a good first serious Colab experiment: more seeds before very large simulation counts.

In [ ]:
repeated = run_repeated_backtests(
    df,
    seeds=list(range(100)),
    training_window=300,
    test_draws=200,
    recent_draw_count=100,
    simulations=100,
    half_life=250,
)

repeated_summary = summarise_repeated_backtests(repeated)

repeated.to_csv(output_dir / 'large_repeated_runs_100_seeds.csv', index=False)
repeated_summary.to_csv(output_dir / 'large_repeated_summary_100_seeds.csv', index=False)

repeated_summary

## 5. Parameter Sweep

Keep this bounded at first. Parameter sweeps multiply quickly.

In [ ]:
sweep = run_parameter_sweep(
    df,
    half_lives=[75, 150, 250, 400],
    simulation_counts=[100, 300],
    training_windows=[300, 500],
    recent_draw_counts=[50, 100],
    test_draws=200,
    random_seed=42,
)

sweep.to_csv(output_dir / 'large_parameter_sweep.csv', index=False)
sweep.head(20)

## 6. Ablation Retest

This checks whether the no-spread result still looks interesting over a larger window.

In [ ]:
ablation = backtest_model_ablation(
    df,
    training_window=300,
    history_window=300,
    test_draws=200,
    simulations=300,
    half_life=250,
    random_seed=42,
)

ablation_summary = summarise_backtest(ablation)
ablation_tier_values = summarise_prize_tier_values(ablation)

ablation.to_csv(output_dir / 'large_ablation_runs.csv', index=False)
ablation_summary.to_csv(output_dir / 'large_ablation_summary.csv', index=False)
ablation_tier_values.to_csv(output_dir / 'large_ablation_tier_values.csv', index=False)

ablation_summary

## Scaling Notes

- Increase seeds before increasing simulations.
- Save CSVs after every long cell.
- If Colab disconnects, rerun setup cells and continue from the saved CSVs in Google Drive.
- Good next steps: compare `half_life=250` vs `400`, and retest the no-spread ablation over more than one seed.